# UCI Heart Disease — Exploratory Data Analysis

Project: P2 — Conformal Prediction · Uncertainty-Aware Medical AI  
Dataset: Cleveland Heart Disease (303 rows, 13 features)  
Target: binarised (0 = no disease, 1 = disease present)

In [ ]:
import matplotlib
matplotlib.use('Agg')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

FIGURES_DIR = Path('../reports/figures/eda')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

df_raw = pd.read_csv(
    '../data/raw/heart.csv',
    header=None,
    names=COLUMNS,
    na_values=['?']
)
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
n_before = len(df_raw)
df = df_raw.dropna().reset_index(drop=True)
print(f'Dropped {n_before - len(df)} NaN rows → {len(df)} rows remain')

for col in COLUMNS:
    if col == 'oldpeak':
        df[col] = df[col].astype(float)
    else:
        df[col] = df[col].astype(int)

# Remap Cleveland codes to 0-indexed (matches CLAUDE.md API schema)
df['cp']    = df['cp'].map({1: 0, 2: 1, 3: 2, 4: 3})
df['slope'] = df['slope'].map({1: 0, 2: 1, 3: 2})
df['thal']  = df['thal'].map({3: 0, 6: 1, 7: 2})
df['target'] = (df['target'] > 0).astype(int)

print('\ndtypes:')
print(df.dtypes)
print('\nTarget distribution:')
print(df['target'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['target'].value_counts().sort_index()
ax.bar(['No Disease (0)', 'Disease (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Target Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'target_distribution.png', dpi=150)
plt.close(fig)
print('Saved target_distribution.png')

In [ ]:
FEATURES = [c for c in COLUMNS if c != 'target']
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes_flat = axes.flatten()

for i, feat in enumerate(FEATURES):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        axes_flat[i].hist(
            df.loc[df['target'] == label, feat],
            bins=20, alpha=0.6, color=color, label=str(label)
        )
    axes_flat[i].set_title(feat)
    axes_flat[i].legend(title='target')

# hide unused subplots
for j in range(len(FEATURES), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Feature Distributions by Target Class', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_distributions.png', dpi=150)
plt.close(fig)
print('Saved feature_distributions.png')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    ax=ax
)
ax.set_title('Feature Correlation Heatmap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
plt.close(fig)
print('Saved correlation_heatmap.png')

## Key Takeaways

- **Class balance**: 165 no-disease (55.6%) vs 138 disease (44.4%) — mild imbalance, weighted F1 is the right primary metric.
- **Top predictors**: `thalach` (max heart rate), `cp` (chest pain type), and `exang` (exercise-induced angina) show the strongest separation between classes.
- **Continuous features**: `age`, `trestbps`, `chol`, and `oldpeak` have right-skewed distributions with some high-value outliers that the StandardScaler will normalise.
- **Correlation**: `cp`, `thalach`, `exang`, `oldpeak`, and `slope` correlate with `target` (|r| > 0.30); `fbs`, `chol`, `restecg` are weak predictors.
- **6 rows dropped** (303 → 297) due to `?` placeholder values in `ca` and `thal` columns.